# CS781 Phase 1 — Resumable decoder-only Transformer training

This version preserves the project's existing Phase-4 data foundation and streaming JSONL pipeline:
- `legal_bpe_32k.json` tokenizer
- train/validation manifests
- byte-offset `source_index.jsonl`
- `WorkerSafeDocumentAwarePacker` (does not concatenate the whole corpus into RAM)
- randomly initialized GPT-2-style decoder model using the original `MODEL_CONFIG`

**Target:** train to optimizer step **3,000** (set `TOTAL_STEPS = 2_000` if you want the shorter target). Local checkpoints are written frequently; the latest checkpoint is uploaded to a private Kaggle Dataset periodically so a new Kaggle session can resume.

**Before running:** attach the existing Phase-4 artifact dataset as notebook input. Create a private Kaggle Dataset named `cs781-phase1-checkpoints` once, with a `dataset-metadata.json` matching the slug below, and add Kaggle Secrets `KAGGLE_USERNAME` and `KAGGLE_KEY`. If the checkpoint dataset already exists, keep it attached as an input too.

Run the cells top-to-bottom. Do not run the old notebook's one-off training-step or hard-coded “resume from step 700” cells.


In [ ]:
# Install only if the environment is missing these packages.
%pip -q install transformers tokenizers


In [ ]:
# ============================================================
# CELL 1 — ENVIRONMENT
# ============================================================

import os
import gc
import json
import math
import time
import random
from pathlib import Path

import torch
from torch.utils.data import IterableDataset, DataLoader, get_worker_info

print("=" * 80)
print("CS781 PHASE 1 — TRAINING ENVIRONMENT")
print("=" * 80)

print("PyTorch :", torch.__version__)
print("CUDA    :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU     :", torch.cuda.get_device_name(0))
    print(
        "VRAM    :",
        f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB"
    )

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device  :", DEVICE)

assert torch.cuda.is_available(), "CUDA GPU is required for this run."

print("\n✓ Environment ready")

In [ ]:
# ============================================================
# CELL 2 — PHASE-1 ARTIFACT PATHS
# ============================================================

CANDIDATES = [
    Path("/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep"),
    Path("/kaggle/input/datasets/oipranesh/cs781-phase4-final-recovery"),
]

CHECKPOINT = None

for candidate in CANDIDATES:
    if (
        (candidate / "manifests/train_manifest.jsonl").exists()
        and
        (candidate / "tokenizer/legal_bpe_32k.json").exists()
    ):
        CHECKPOINT = candidate
        break

assert CHECKPOINT is not None, "Phase-1 training artifacts were not found."

TRAIN_MANIFEST = CHECKPOINT / "manifests/train_manifest.jsonl"
VAL_MANIFEST = CHECKPOINT / "manifests/validation_manifest.jsonl"
TEST_MANIFEST = CHECKPOINT / "manifests/test_manifest.jsonl"
TOKENIZER_PATH = CHECKPOINT / "tokenizer/legal_bpe_32k.json"

print("=" * 80)
print("PHASE-1 ARTIFACTS")
print("=" * 80)

print("Checkpoint :", CHECKPOINT)
print("Train      :", TRAIN_MANIFEST)
print("Validation :", VAL_MANIFEST)
print("Test       :", TEST_MANIFEST)
print("Tokenizer  :", TOKENIZER_PATH)

assert TRAIN_MANIFEST.exists()
assert VAL_MANIFEST.exists()
assert TEST_MANIFEST.exists()
assert TOKENIZER_PATH.exists()

print("\n✓ All required artifacts found")

In [ ]:
# ============================================================
# CELL 3 — TOKENIZER
# ============================================================

from tokenizers import Tokenizer

tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))

VOCAB_SIZE = tokenizer.get_vocab_size()

PAD_ID = tokenizer.token_to_id("[PAD]")
UNK_ID = tokenizer.token_to_id("[UNK]")
BOS_ID = tokenizer.token_to_id("[BOS]")
EOS_ID = tokenizer.token_to_id("[EOS]")

print("=" * 80)
print("TOKENIZER")
print("=" * 80)

print("Vocabulary :", VOCAB_SIZE)
print("PAD        :", PAD_ID)
print("UNK        :", UNK_ID)
print("BOS        :", BOS_ID)
print("EOS        :", EOS_ID)

assert VOCAB_SIZE == 32000
assert PAD_ID is not None
assert BOS_ID is not None
assert EOS_ID is not None

print("\n✓ Tokenizer loaded")
print("✓ Vocabulary = 32,000")

In [ ]:
# ============================================================
# CELL 4 — STREAMING SOURCE INDEX
# ============================================================

INDEX_CANDIDATES = [
    Path("/kaggle/working/phase4_streaming_index/train_source_index.jsonl"),
    Path(
        "/kaggle/input/datasets/oipranesh/"
        "cs781-phase4-final-recovery/indexes/source_index.jsonl"
    ),
    CHECKPOINT / "indexes/source_index.jsonl",
]

INDEX_PATH = None

for candidate in INDEX_CANDIDATES:
    if candidate.exists():
        INDEX_PATH = candidate
        break

if INDEX_PATH is not None:

    print("Existing source index found:")
    print(INDEX_PATH)

else:

    print("No existing source index found.")
    print("Building index from the Phase-1 deduplicated corpus...")

    DEDUP_ROOT = CHECKPOINT / "dedup"

    assert DEDUP_ROOT.exists(), DEDUP_ROOT

    INDEX_DIR = Path("/kaggle/working/phase4_streaming_index")
    INDEX_DIR.mkdir(parents=True, exist_ok=True)

    INDEX_PATH = INDEX_DIR / "train_source_index.jsonl"

    # Discover source files referenced by the training manifest.
    source_names = set()

    with open(TRAIN_MANIFEST, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rec = json.loads(line)
                source_names.add(rec["source_file"])

    print("Manifest source files:", len(source_names))

    def resolve_source(source_file):
        candidates = [
            DEDUP_ROOT / source_file,
            DEDUP_ROOT / Path(source_file).with_suffix(""),
        ]

        for p in candidates:
            if p.exists() and p.is_file():
                return p

        matches = [
            p for p in DEDUP_ROOT.rglob(Path(source_file).name)
            if p.is_file()
        ]

        if len(matches) == 1:
            return matches[0]

        raise FileNotFoundError(
            f"Could not resolve source file: {source_file}"
        )

    source_paths = {
        name: resolve_source(name)
        for name in source_names
    }

    print("\nBuilding byte-offset index...")

    total_records = 0

    with open(INDEX_PATH, "w", encoding="utf-8") as out:

        for source_name, source_path in source_paths.items():

            print("Indexing:", source_name)

            with open(
                source_path,
                "rb",
                buffering=1024 * 1024,
            ) as src:

                while True:

                    offset = src.tell()
                    raw = src.readline()

                    if not raw:
                        break

                    try:
                        obj = json.loads(raw)
                    except Exception as e:
                        raise RuntimeError(
                            f"Invalid JSON at {source_path}:{offset}"
                        ) from e

                    text = obj.get("text")

                    if not isinstance(text, str):
                        continue

                    text_hash = __import__("hashlib").sha256(
                        text.encode("utf-8")
                    ).hexdigest()

                    out.write(
                        json.dumps(
                            {
                                "text_hash": text_hash,
                                "source_file": source_name,
                                "byte_offset": offset,
                                "byte_length": len(raw),
                            },
                            separators=(",", ":"),
                        )
                        + "\n"
                    )

                    total_records += 1

            print("  indexed")

    print("\nTotal indexed records:", f"{total_records:,}")

print("\nIndex:", INDEX_PATH)
assert INDEX_PATH.exists()

print("✓ Source index ready")

In [ ]:
# ============================================================
# CELL 5 — WORKER-SAFE DOCUMENT-AWARE STREAMING DATASET
# ============================================================

class WorkerSafeDocumentAwarePacker(IterableDataset):

    def __init__(
        self,
        manifest_path,
        index_path,
        tokenizer,
        context_length=4096,
        checkpoint=None,
        split="train",
    ):
        super().__init__()

        self.manifest_path = str(manifest_path)
        self.index_path = str(index_path)
        self.tokenizer = tokenizer
        self.context_length = int(context_length)
        self.checkpoint = str(checkpoint or CHECKPOINT)
        self.split = split

        # ----------------------------------------------------
        # Load byte-offset index
        # ----------------------------------------------------

        self.index = {}

        with open(self.index_path, "r", encoding="utf-8") as f:

            for line in f:

                if not line.strip():
                    continue

                rec = json.loads(line)

                self.index[rec["text_hash"]] = {
                    "source_file": rec["source_file"],
                    "byte_offset": rec["byte_offset"],
                    "byte_length": rec["byte_length"],
                }

        # ----------------------------------------------------
        # Load manifest
        # ----------------------------------------------------

        self.manifest = []

        with open(self.manifest_path, "r", encoding="utf-8") as f:

            for line in f:

                if line.strip():
                    self.manifest.append(json.loads(line))

        # ----------------------------------------------------
        # Special tokens
        # ----------------------------------------------------

        self.bos_id = int(tokenizer.token_to_id("[BOS]"))
        self.eos_id = int(tokenizer.token_to_id("[EOS]"))
        self.pad_id = int(tokenizer.token_to_id("[PAD]"))

        print(
            f"{split}: "
            f"{len(self.manifest):,} manifest records | "
            f"{len(self.index):,} index records"
        )

    def _read_source_record(self, rec):

        text_hash = rec["text_hash"]

        if text_hash not in self.index:
            raise KeyError(
                f"text_hash not found: {text_hash}"
            )

        idx = self.index[text_hash]

        source_name = idx["source_file"]

        if source_name.endswith(".gz"):
            source_name = source_name[:-3]

        source_path = (
            Path(self.checkpoint)
            / "dedup"
            / source_name
        )

        if not source_path.exists():

            # Robust fallback for unusual artifact layout.
            matches = list(
                Path(self.checkpoint)
                .rglob(Path(source_name).name)
            )

            if len(matches) != 1:
                raise FileNotFoundError(
                    f"Cannot resolve source:\n{source_name}"
                )

            source_path = matches[0]

        with open(source_path, "rb") as f:

            f.seek(idx["byte_offset"])
            raw = f.read(idx["byte_length"])

        raw = raw.rstrip(b"\r\n")

        return json.loads(raw.decode("utf-8"))

    def _extract_text(self, record):

        if isinstance(record, str):
            return record

        for key in ("text", "content", "document", "body"):

            value = record.get(key)

            if isinstance(value, str):
                return value

        raise KeyError(
            f"No text field found. Keys: {list(record.keys())}"
        )

    def _tokenize_document(self, text):

        ids = self.tokenizer.encode(text).ids

        return (
            [self.bos_id]
            + list(map(int, ids))
            + [self.eos_id]
        )

    def _pack_document(self, token_ids):

        for start in range(
            0,
            len(token_ids),
            self.context_length,
        ):

            chunk = token_ids[
                start:start + self.context_length
            ]

            if len(chunk) < self.context_length:

                chunk = chunk + [
                    self.pad_id
                ] * (
                    self.context_length - len(chunk)
                )

            yield chunk

    def __iter__(self):

        worker_info = get_worker_info()

        if worker_info is None:

            worker_id = 0
            num_workers = 1

        else:

            worker_id = worker_info.id
            num_workers = worker_info.num_workers

        # Strided worker partitioning.
        for manifest_idx in range(
            worker_id,
            len(self.manifest),
            num_workers,
        ):

            rec = self.manifest[manifest_idx]

            source_record = self._read_source_record(rec)

            text = self._extract_text(source_record)

            token_ids = self._tokenize_document(text)

            for packed in self._pack_document(token_ids):

                input_ids = torch.tensor(
                    packed,
                    dtype=torch.long,
                )

                labels = input_ids.clone()

                labels[
                    labels == self.pad_id
                ] = -100

                attention_mask = (
                    input_ids != self.pad_id
                ).long()

                yield {
                    "input_ids": input_ids,
                    "labels": labels,
                    "attention_mask": attention_mask,
                }


print("✓ Worker-safe streaming dataset class ready")

In [ ]:
# ============================================================
# CELL 6 — TRAIN / VALIDATION DATASETS + DATALOADERS
# ============================================================

CONTEXT_LENGTH = 4096

BATCH_SIZE = 1
NUM_WORKERS = 0

train_dataset = WorkerSafeDocumentAwarePacker(
    manifest_path=TRAIN_MANIFEST,
    index_path=INDEX_PATH,
    tokenizer=tokenizer,
    context_length=CONTEXT_LENGTH,
    checkpoint=CHECKPOINT,
    split="train",
)

validation_dataset = WorkerSafeDocumentAwarePacker(
    manifest_path=VAL_MANIFEST,
    index_path=INDEX_PATH,
    tokenizer=tokenizer,
    context_length=CONTEXT_LENGTH,
    checkpoint=CHECKPOINT,
    split="validation",
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=False,
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=False,
)

print("=" * 80)
print("DATALOADERS READY")
print("=" * 80)

print("Train records :", f"{len(train_dataset.manifest):,}")
print("Val records   :", f"{len(validation_dataset.manifest):,}")
print("Batch size    :", BATCH_SIZE)
print("Workers       :", NUM_WORKERS)
print("Context       :", CONTEXT_LENGTH)

In [ ]:
# ============================================================
# MODEL CONFIGURATION
# ============================================================

from dataclasses import dataclass

# Use the tokenizer that was already loaded, if available.
# Otherwise load it directly from the Phase-4 artifact.
if "tokenizer" not in globals():
    from tokenizers import Tokenizer

    TOKENIZER_PATH = (
        "/kaggle/input/datasets/oipranesh/"
        "cs781-phase4-training-prep/tokenizer/legal_bpe_32k.json"
    )

    tokenizer = Tokenizer.from_file(TOKENIZER_PATH)

# Resolve special-token IDs directly from the tokenizer.
VOCAB_SIZE = tokenizer.get_vocab_size()

PAD_ID = tokenizer.token_to_id("[PAD]")
BOS_ID = tokenizer.token_to_id("[BOS]")
EOS_ID = tokenizer.token_to_id("[EOS]")

print("Vocabulary:", VOCAB_SIZE)
print("PAD_ID:", PAD_ID)
print("BOS_ID:", BOS_ID)
print("EOS_ID:", EOS_ID)

assert VOCAB_SIZE == 32000
assert PAD_ID is not None
assert BOS_ID is not None
assert EOS_ID is not None


@dataclass
class LegalLMConfig:
    vocab_size: int = VOCAB_SIZE
    max_position_embeddings: int = 4096

    hidden_size: int = 768
    intermediate_size: int = 3072

    num_hidden_layers: int = 12
    num_attention_heads: int = 12

    hidden_dropout: float = 0.0
    attention_dropout: float = 0.0

    bos_token_id: int = BOS_ID
    eos_token_id: int = EOS_ID
    pad_token_id: int = PAD_ID


MODEL_CONFIG = LegalLMConfig()

print("\nModel configuration:")
print(MODEL_CONFIG)

In [ ]:
# ============================================================
# CREATE RANDOMLY INITIALIZED DECODER-ONLY TRANSFORMER
# ============================================================

from transformers import GPT2Config, GPT2LMHeadModel

hf_config = GPT2Config(
    vocab_size=MODEL_CONFIG.vocab_size,
    n_positions=MODEL_CONFIG.max_position_embeddings,
    n_ctx=MODEL_CONFIG.max_position_embeddings,

    n_embd=MODEL_CONFIG.hidden_size,
    n_layer=MODEL_CONFIG.num_hidden_layers,
    n_head=MODEL_CONFIG.num_attention_heads,
    n_inner=MODEL_CONFIG.intermediate_size,

    resid_pdrop=MODEL_CONFIG.hidden_dropout,
    embd_pdrop=MODEL_CONFIG.hidden_dropout,
    attn_pdrop=MODEL_CONFIG.attention_dropout,

    bos_token_id=MODEL_CONFIG.bos_token_id,
    eos_token_id=MODEL_CONFIG.eos_token_id,
    pad_token_id=MODEL_CONFIG.pad_token_id,

    use_cache=False,
)

# Fresh model from configuration.
# NO pretrained checkpoint is loaded.
model = GPT2LMHeadModel(hf_config)

print("✓ Model created from scratch")

In [ ]:
# ============================================================
# PARAMETER CHECK
# ============================================================

param_count = sum(p.numel() for p in model.parameters())

print(f"Parameter count: {param_count:,}")

assert 50_000_000 <= param_count <= 300_000_000

print("✓ Parameter count is within Phase-1 requirement")

In [ ]:
# ============================================================
# CELL 9 — GPU + MEMORY OPTIMIZATION
# ============================================================

model = model.to(DEVICE)

model.gradient_checkpointing_enable()
model.config.use_cache = False

print("Device :", next(model.parameters()).device)
print("Gradient checkpointing : enabled")
print("use_cache              :", model.config.use_cache)

assert next(model.parameters()).is_cuda

print("✓ Model ready for T4 training")

In [ ]:
# ============================================================
# TRAINING CONFIGURATION — resumable run
# ============================================================
import os, json, math, time, shutil, signal, subprocess
from pathlib import Path

# Set to 2_000 for a shorter run; 3_000 is the intended target.
TOTAL_STEPS = 3_000
LEARNING_RATE = 5e-5
WEIGHT_DECAY = 0.01
GRAD_ACCUM = 8
MAX_GRAD_NORM = 1.0
LOG_EVERY = 10
LOCAL_CHECKPOINT_EVERY = 25
KAGGLE_UPLOAD_EVERY = 250
NUM_WORKERS = 2
BATCH_SIZE = 1
CONTEXT_LENGTH = 4096

OUTPUT_DIR = Path("/kaggle/working/phase1_decoder_transformer")
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = OUTPUT_DIR / "training_log.jsonl"
STATE_PATH = OUTPUT_DIR / "training_state.json"
LATEST_PATH = CHECKPOINT_DIR / "checkpoint_latest.pt"

# Kaggle Dataset slug must already exist for `kaggle datasets version`.
KAGGLE_DATASET_SLUG = "oipranesh/cs781-phase-1-checkpoints"
UPLOAD_DIR = Path("/kaggle/working/phase1_checkpoint_dataset")
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

# Pull Kaggle API credentials from Kaggle Secrets, if configured.
try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    os.environ["KAGGLE_USERNAME"] = _secrets.get_secret("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = _secrets.get_secret("KAGGLE_KEY")
    print("Kaggle API credentials loaded from Secrets.")
except Exception as e:
    print("Kaggle Secrets not available in this session:", repr(e))
    print("Training can still run locally; Dataset uploads will be skipped until credentials are configured.")

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)
scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())
USE_AMP = torch.cuda.is_available()
AMP_DTYPE = torch.float16

print("Total target steps:", TOTAL_STEPS)
print("Local checkpoint every:", LOCAL_CHECKPOINT_EVERY)
print("Kaggle upload every:", KAGGLE_UPLOAD_EVERY)
print("Output:", OUTPUT_DIR)


In [ ]:
# ============================================================
# CHECKPOINT DISCOVERY + SAFE SAVE + KAGGLE DATASET VERSIONING
# ============================================================
import re

def find_resume_checkpoint():
    # Prefer rolling checkpoint files; use filename step numbers for snapshots
    # so we do not load multi-GB optimizer states merely to inspect their steps.
    candidates = []
    if LATEST_PATH.exists():
        candidates.append((10**18, LATEST_PATH))
    for p in Path("/kaggle/input").rglob("checkpoint_latest.pt"):
        candidates.append((10**18, p))
    for p in Path("/kaggle/input").rglob("checkpoint_step_*.pt"):
        match = re.search(r"checkpoint_step_(\\d+)\\.pt$", p.name)
        if match:
            candidates.append((int(match.group(1)), p))
    if not candidates:
        return None
    # For rolling checkpoint(s), inspect their saved step once each. Numbered
    # snapshots already carry a step in the filename.
    ranked = []
    for named_step, p in candidates:
        if named_step == 10**18:
            try:
                ck = torch.load(p, map_location="cpu", weights_only=False)
                step = int(ck.get("step", -1))
                if step >= 0 and "model_state_dict" in ck:
                    ranked.append((step, p))
            except Exception as e:
                print("Ignoring unreadable checkpoint:", p, repr(e))
        else:
            ranked.append((named_step, p))
    if not ranked:
        return None
    ranked.sort(key=lambda x: x[0])
    step, path = ranked[-1]
    print(f"Found resume checkpoint at step {step}: {path}")
    return path

def save_checkpoint(step, reason="periodic"):
    temp_path = CHECKPOINT_DIR / "checkpoint_tmp.pt"
    payload = {
        "step": int(step),
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "model_config": {
            "vocab_size": MODEL_CONFIG.vocab_size,
            "context_length": MODEL_CONFIG.max_position_embeddings,
            "hidden_size": MODEL_CONFIG.hidden_size,
            "num_layers": MODEL_CONFIG.num_hidden_layers,
            "num_heads": MODEL_CONFIG.num_attention_heads,
        },
        "reason": reason,
        "saved_unix_time": time.time(),
    }
    torch.save(payload, temp_path)
    os.replace(temp_path, LATEST_PATH)
    # Keep a numbered snapshot alongside the rolling checkpoint.
    numbered = CHECKPOINT_DIR / f"checkpoint_step_{step:06d}.pt"
    shutil.copy2(LATEST_PATH, numbered)
    # Retain only the two most recent numbered snapshots to limit disk use.
    snapshots = sorted(CHECKPOINT_DIR.glob("checkpoint_step_*.pt"))
    for old in snapshots[:-2]:
        try: old.unlink()
        except OSError: pass
    with open(STATE_PATH, "w", encoding="utf-8") as f:
        json.dump({"step": int(step), "reason": reason, "saved_unix_time": time.time()}, f, indent=2)
    print(f"Saved checkpoint: step={step}, size={LATEST_PATH.stat().st_size / 1e9:.2f} GB ({reason})")

def upload_checkpoint_to_kaggle(step):
    if not (os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY")):
        print("Skipping Dataset upload: Kaggle API credentials are not configured.")
        return False
    if not (UPLOAD_DIR / "dataset-metadata.json").exists():
        metadata = {
            "title": "CS781 Phase 1 Training Checkpoints",
            "id": KAGGLE_DATASET_SLUG,
            "licenses": [{"name": "CC0-1.0"}],
            "private": True,
        }
        (UPLOAD_DIR / "dataset-metadata.json").write_text(json.dumps(metadata, indent=2))
    shutil.copy2(LATEST_PATH, UPLOAD_DIR / "checkpoint_latest.pt")
    if LOG_PATH.exists():
        shutil.copy2(LOG_PATH, UPLOAD_DIR / "training_log.jsonl")
    (UPLOAD_DIR / "training_state.json").write_text(json.dumps({
        "step": int(step), "total_steps": int(TOTAL_STEPS), "updated_unix_time": time.time()
    }, indent=2))
    try:
        result = subprocess.run(
            ["kaggle", "datasets", "version", "-p", str(UPLOAD_DIR),
             "-m", f"CS781 Phase 1 checkpoint step {step}", "--dir-mode", "tar"],
            check=True, text=True, capture_output=True, timeout=1800,
        )
        print(result.stdout[-3000:] if result.stdout else "Kaggle Dataset version uploaded.")
        return True
    except Exception as e:
        print("Kaggle upload failed (local checkpoint is safe):", repr(e))
        if getattr(e, "stdout", None): print(e.stdout[-2000:])
        if getattr(e, "stderr", None): print(e.stderr[-3000:])
        return False

# Resume only if a checkpoint is actually attached/found.
# Model architecture must match the checkpoint's recorded configuration.
RESUME_PATH = find_resume_checkpoint()
START_STEP = 0
if RESUME_PATH is not None:
    checkpoint = torch.load(RESUME_PATH, map_location=DEVICE, weights_only=False)
    saved_cfg = checkpoint.get("model_config", {})
    current_cfg = {
        "vocab_size": MODEL_CONFIG.vocab_size,
        "context_length": MODEL_CONFIG.max_position_embeddings,
        "hidden_size": MODEL_CONFIG.hidden_size,
        "num_layers": MODEL_CONFIG.num_hidden_layers,
        "num_heads": MODEL_CONFIG.num_attention_heads,
    }
    if saved_cfg and saved_cfg != current_cfg:
        raise ValueError(f"Checkpoint architecture mismatch. Saved={saved_cfg}; current={current_cfg}")
    model.load_state_dict(checkpoint["model_state_dict"])
    if "optimizer_state_dict" in checkpoint:
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    if "scaler_state_dict" in checkpoint:
        scaler.load_state_dict(checkpoint["scaler_state_dict"])
    START_STEP = int(checkpoint["step"])
    print("Resumed from optimizer step:", START_STEP)
else:
    print("No checkpoint found: starting from randomly initialized model at step 0.")

assert 0 <= START_STEP <= TOTAL_STEPS


In [ ]:
# ============================================================
# TRAIN TO TOTAL_STEPS
# ============================================================
model.train()
train_iterator = iter(train_loader)
running_loss = 0.0
logged_batches = 0
run_started = time.time()

def _save_on_termination(signum, frame):
    print(f"Received signal {signum}; attempting emergency checkpoint save.")
    try:
        save_checkpoint(current_step, reason=f"signal_{signum}")
    except Exception as e:
        print("Emergency save failed:", repr(e))
    raise SystemExit(128 + signum)

# Best effort only; Kaggle may terminate a session without delivering a signal.
try:
    signal.signal(signal.SIGTERM, _save_on_termination)
    signal.signal(signal.SIGINT, _save_on_termination)
except Exception:
    pass

current_step = START_STEP
for step in range(START_STEP + 1, TOTAL_STEPS + 1):
    current_step = step
    optimizer.zero_grad(set_to_none=True)
    accumulated_loss = 0.0

    for micro_step in range(GRAD_ACCUM):
        try:
            batch = next(train_iterator)
        except StopIteration:
            train_iterator = iter(train_loader)
            batch = next(train_iterator)

        input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
        attention_mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
        labels = batch["labels"].to(DEVICE, non_blocking=True)

        with torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            scaled_loss = loss / GRAD_ACCUM

        scaler.scale(scaled_loss).backward()
        accumulated_loss += float(loss.detach().float().item())

    scaler.unscale_(optimizer)
    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
    scaler.step(optimizer)
    scaler.update()

    mean_loss = accumulated_loss / GRAD_ACCUM
    running_loss += mean_loss
    logged_batches += 1

    if step % LOG_EVERY == 0 or step == START_STEP + 1:
        avg_loss = running_loss / max(logged_batches, 1)
        record = {
            "step": step, "loss": avg_loss,
            "perplexity": math.exp(min(avg_loss, 20)),
            "grad_norm": float(grad_norm),
            "elapsed_seconds": time.time() - run_started,
        }
        with open(LOG_PATH, "a", encoding="utf-8") as f:
            f.write(json.dumps(record) + "\n")
        print(f"step={step:5d}/{TOTAL_STEPS} | loss={avg_loss:.4f} | ppl={record['perplexity']:.2f} | grad={float(grad_norm):.3f} | elapsed={(time.time()-run_started)/60:.1f} min")
        running_loss = 0.0
        logged_batches = 0

    if step % LOCAL_CHECKPOINT_EVERY == 0 or step == TOTAL_STEPS:
        save_checkpoint(step, reason="final" if step == TOTAL_STEPS else "periodic")

    if step % KAGGLE_UPLOAD_EVERY == 0 or step == TOTAL_STEPS:
        # Save immediately before upload. If upload fails, the local checkpoint remains.
        if step % LOCAL_CHECKPOINT_EVERY != 0 and step != TOTAL_STEPS:
            save_checkpoint(step, reason="upload_boundary")
        upload_checkpoint_to_kaggle(step)

print("\nTraining loop finished.")
print("Final step:", current_step)
print("Latest checkpoint:", LATEST_PATH)


In [ ]:
from pathlib import Path

ROOTS = [Path("/kaggle/input"), Path("/kaggle/working")]

def find_files(names):
    found = []
    for root in ROOTS:
        if root.exists():
            for name in names:
                found.extend(root.rglob(name))
    return list(dict.fromkeys(p for p in found if p.is_file()))

checkpoints = find_files(["checkpoint_latest.pt"])
tests = find_files(["test.json", "Phase1- Test.json"])
tokenizer_configs = find_files(["tokenizer.json"])
tokenizer_vocab = find_files(["vocab.json"])

print("CHECKPOINTS:")
for p in checkpoints:
    print(p, f"({p.stat().st_size / 1e9:.2f} GB)")

print("\nTEST FILES:")
for p in tests:
    print(p)

print("\nTOKENIZER FILES:")
for p in tokenizer_configs + tokenizer_vocab:
    print(p)

In [ ]:

import json
import torch
from pathlib import Path

CHECKPOINT_PATH = Path(
    "/kaggle/input/datasets/oipranesh/cs781-phase-1-checkpoints/checkpoint_latest.pt"
)
TEST_PATH = Path(
    "/kaggle/input/datasets/shubhamjoshi1806/phase1-test-json/test.json"
)
TOKENIZER_PATH = Path(
    "/kaggle/input/datasets/oipranesh/cs781-phase4-training-prep/tokenizer/legal_bpe_32k.json"
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Verify files exist
for path in [CHECKPOINT_PATH, TEST_PATH, TOKENIZER_PATH]:
    print(f"{path}: exists={path.is_file()}")
    if not path.is_file():
        raise FileNotFoundError(path)

# Inspect test data
with open(TEST_PATH, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print("\nTest type:", type(test_data).__name__)
print("Number of examples:", len(test_data))

first_key = next(iter(test_data))
print("First ID:", first_key)
print("First record:", str(test_data[first_key])[:1000])

# Load checkpoint
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

print("\nCheckpoint keys:", list(checkpoint.keys()))
print("Saved training step:", checkpoint.get("step"))
print("Model config:", checkpoint.get("model_config"))

# Inspect tokenizer JSON structure
with open(TOKENIZER_PATH, "r", encoding="utf-8") as f:
    tokenizer_json = json.load(f)

print("\nTokenizer JSON top-level keys:", list(tokenizer_json.keys()))
print("Tokenizer model type:",
      tokenizer_json.get("model", {}).get("type"))


In [ ]:
from tokenizers import Tokenizer, decoders

bpe = Tokenizer.from_file(str(TOKENIZER_PATH))
bpe.decoder = decoders.ByteLevel()

PAD_ID = bpe.token_to_id("[PAD]")
BOS_ID = bpe.token_to_id("[BOS]")
EOS_ID = bpe.token_to_id("[EOS]")

assert bpe.get_vocab_size() == 32000
assert PAD_ID is not None
assert BOS_ID is not None
assert EOS_ID is not None

print("Vocabulary:", bpe.get_vocab_size())
print("PAD/BOS/EOS:", PAD_ID, BOS_ID, EOS_ID)

test_text = "The Court shall consider the matter."
print("Round-trip:", bpe.decode(bpe.encode(test_text).ids))

In [ ]:
import torch

CONTEXT_LENGTH = 4096
MAX_GENERATION_TOKENS = 300

def generate_128_words(prompt, temperature=0.8, top_k=50, seed=42):
    encoded = bpe.encode(prompt)
    prompt_ids = encoded.ids

    if not prompt_ids:
        prompt_ids = [BOS_ID]

    # Reserve room for generated tokens.
    prompt_ids = prompt_ids[-(CONTEXT_LENGTH - MAX_GENERATION_TOKENS):]
    input_ids = torch.tensor(
        [prompt_ids], dtype=torch.long, device=device
    )
    generated = input_ids.clone()

    # Use a local generator so each example has reproducible sampling.
    rng = torch.Generator(device=device)
    rng.manual_seed(seed)

    with torch.inference_mode():
        for _ in range(MAX_GENERATION_TOKENS):
            context_ids = generated[:, -CONTEXT_LENGTH:]

            outputs = model(
                input_ids=context_ids,
                attention_mask=torch.ones_like(context_ids),
                use_cache=False,
            )
            logits = outputs.logits[:, -1, :] / temperature

            if top_k is not None:
                k = min(top_k, logits.shape[-1])
                values, indices = torch.topk(logits, k, dim=-1)
                filtered = torch.full_like(logits, float("-inf"))
                filtered.scatter_(1, indices, values)
                logits = filtered

            probs = torch.softmax(logits, dim=-1)
            next_token = torch.multinomial(
                probs, num_samples=1, generator=rng
            )
            next_id = next_token.item()

            if next_id == EOS_ID:
                break

            generated = torch.cat([generated, next_token], dim=1)

            new_ids = generated[0, input_ids.shape[1]:].tolist()
            text = bpe.decode(new_ids, skip_special_tokens=True).strip()

            if len(text.split()) >= 128:
                break

    new_ids = generated[0, input_ids.shape[1]:].tolist()
    text = bpe.decode(new_ids, skip_special_tokens=True).strip()
    return " ".join(text.split()[:128])

In [ ]:
first_id = next(iter(test_data))

sample_output = generate_128_words(
    test_data[first_id]["text"],
    seed=42,
)

print("ID:", first_id)
print("Generated words:", len(sample_output.split()))
print(sample_output)

assert "Ġ" not in sample_output
assert "Ċ" not in sample_output

In [ ]:
import json
import time
from pathlib import Path

output_path = Path("/kaggle/working/prediction.json")
predictions = {}
start_time = time.time()
total = len(test_data)

for i, (test_id, item) in enumerate(test_data.items(), start=1):
    predictions[str(test_id)] = {
        "text": generate_128_words(
            item["text"],
            seed=42 + i,
        )
    }

    if i % 10 == 0 or i == total:
        print(f"[{i}/{total}] ID={test_id}")

with output_path.open("w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

print("Saved:", output_path)
print("Examples:", len(predictions))
print(f"Elapsed: {(time.time() - start_time) / 60:.1f} minutes")

In [ ]:
with output_path.open("r", encoding="utf-8") as f:
    saved = json.load(f)

assert len(saved) == len(test_data)
assert list(saved.keys()) == [str(k) for k in test_data.keys()]

for test_id, item in saved.items():
    assert set(item) == {"text"}
    assert isinstance(item["text"], str)
    assert len(item["text"].split()) <= 128
    assert "Ġ" not in item["text"]
    assert "Ċ" not in item["text"]

print("Validation passed.")
print("Output file:", output_path)
print("Size:", f"{output_path.stat().st_size / 1024:.1f} KB")

## After a Kaggle session ends

1. Check the notebook output for the latest successful `Kaggle Dataset version uploaded` message and step number.
2. In the next session, attach the private `cs781-phase1-checkpoints` Dataset as an input.
3. Run this notebook from the top. It searches attached inputs for `checkpoint_latest.pt`, validates the architecture, and restores model, optimizer, scaler, and step.
4. If the session stops between uploads, you will resume from the most recent uploaded checkpoint, not necessarily the last local step.

Kaggle can terminate a session abruptly; no notebook can guarantee an upload at the exact moment of termination. The frequent local saves plus periodic Dataset versions reduce lost work.


In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")


In [ ]:
# CELL 1 — Hugging Face authentication + Phase 2 file discovery

%pip -q install -U huggingface_hub

from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, hf_hub_download, login

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN is empty. Check Kaggle Secrets.")

login(token=HF_TOKEN, add_to_git_credential=False)

HF_REPO = "Exploration-Lab/CS-781-Capstone"
api = HfApi(token=HF_TOKEN)

repo_files = api.list_repo_files(
    repo_id=HF_REPO,
    repo_type="dataset",
)

phase2_files = [
    name for name in repo_files
    if "phase_2_lner" in name.lower()
]

print("Authenticated to Hugging Face.")
print("Phase 2 files discovered:", len(phase2_files))

for name in phase2_files:
    print(name)

assert phase2_files, (
    "No Phase_2_LNER files found. Check repository access and folder names."
)

In [ ]:
# CELL 2 — Download Phase 2 files using the authenticated HF token

import json
from pathlib import Path

PHASE2_DIR = Path("/kaggle/working/phase2_lner")
PHASE2_DIR.mkdir(parents=True, exist_ok=True)

REQUIRED_FILES = [
    "fold1-text.json",
    "fold1-gold.json",
    "fold2-text.json",
    "fold2-gold.json",
    "fold3-text.json",
    "labels.txt",
]

def find_repo_path(filename):
    matches = [
        name for name in phase2_files
        if Path(name).name.lower() == filename.lower()
    ]
    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected one repository file named {filename}; "
            f"found: {matches}"
        )
    return matches[0]

downloaded = {}

for filename in REQUIRED_FILES:
    repo_path = find_repo_path(filename)
    local_path = hf_hub_download(
        repo_id=HF_REPO,
        repo_type="dataset",
        filename=repo_path,
        token=HF_TOKEN,
        local_dir=str(PHASE2_DIR),
    )
    downloaded[filename] = Path(local_path)
    print(f"{filename} -> {local_path}")

def load_json(filename):
    with downloaded[filename].open("r", encoding="utf-8") as f:
        return json.load(f)

train_text_1 = load_json("fold1-text.json")
train_gold_1 = load_json("fold1-gold.json")
train_text_2 = load_json("fold2-text.json")
train_gold_2 = load_json("fold2-gold.json")
test_text = load_json("fold3-text.json")

with downloaded["labels.txt"].open("r", encoding="utf-8") as f:
    ENTITY_LABELS = [line.strip() for line in f if line.strip()]

print("\n=== Dataset summary ===")
print("Fold 1 texts:", len(train_text_1))
print("Fold 1 gold:", len(train_gold_1))
print("Fold 2 texts:", len(train_text_2))
print("Fold 2 gold:", len(train_gold_2))
print("Fold 3 test texts:", len(test_text))
print("Labels:", ENTITY_LABELS)

assert len(train_text_1) == len(train_gold_1), "Fold 1 count mismatch"
assert len(train_text_2) == len(train_gold_2), "Fold 2 count mismatch"
assert test_text, "Fold 3 test data is empty"
assert ENTITY_LABELS, "Label list is empty"

In [ ]:
# CELL 3 — Inspect actual annotation schema

def show_example(texts, gold, fold_name):
    doc_id = next(iter(texts))
    text = texts[doc_id]
    annotation = gold[doc_id]

    print(f"\n=== {fold_name} example ===")
    print("Document ID:", doc_id)
    print("Text type:", type(text).__name__)
    print("Text length:", len(text))
    print("Text preview:", repr(text[:500]))
    print("Annotation type:", type(annotation).__name__)
    print("Raw annotation:", repr(annotation)[:3000])

show_example(train_text_1, train_gold_1, "Fold 1")
show_example(train_text_2, train_gold_2, "Fold 2")

print("\n=== Test example ===")
test_id = next(iter(test_text))
print("Document ID:", test_id)
print("Text length:", len(test_text[test_id]))
print("Text preview:", repr(test_text[test_id][:500]))

In [ ]:
# CELL 4 — Load the Phase 1 tokenizer

from tokenizers import Tokenizer, decoders

TOKENIZER_PATH = Path(
    "/kaggle/input/datasets/oipranesh/"
    "cs781-phase4-training-prep/tokenizer/legal_bpe_32k.json"
)

if not TOKENIZER_PATH.exists():
    raise FileNotFoundError(
        f"Tokenizer not found: {TOKENIZER_PATH}. "
        "Attach your Phase 1 tokenizer dataset to the notebook."
    )

tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH))
tokenizer.decoder = decoders.ByteLevel()

print("Tokenizer vocabulary:", tokenizer.get_vocab_size())
print("Sample tokenization:", tokenizer.encode(
    "The Supreme Court considered the appeal."
).ids[:30])

assert tokenizer.get_vocab_size() == 32000

In [ ]:
# CELL 5 — Validate gold spans and build label IDs

from collections import Counter

assert set(train_text_1) == set(train_gold_1)
assert set(train_text_2) == set(train_gold_2)

all_labels = set(ENTITY_LABELS)
label_counts = Counter()
bad_spans = []
boundary_mismatches = []

def check_fold(texts, gold, fold_name):
    for doc_id, entities in gold.items():
        text = texts[doc_id]

        for ent in entities:
            start, end, label = int(ent[0]), int(ent[1]), str(ent[2])
            entity_text = ent[3] if len(ent) > 3 else None

            if label not in all_labels:
                bad_spans.append((fold_name, doc_id, ent, "unknown label"))
                continue

            if not (0 <= start < end <= len(text)):
                bad_spans.append((fold_name, doc_id, ent, "invalid offsets"))
                continue

            if entity_text is not None and text[start:end] != entity_text:
                boundary_mismatches.append(
                    (fold_name, doc_id, start, end, entity_text, text[start:end])
                )

            label_counts[label] += 1

check_fold(train_text_1, train_gold_1, "fold1")
check_fold(train_text_2, train_gold_2, "fold2")

print("Entity counts by label:")
for label, count in label_counts.most_common():
    print(f"{label:12s} {count:,}")

print("\nInvalid spans:", len(bad_spans))
print("Text/offset mismatches:", len(boundary_mismatches))

if bad_spans:
    print("First invalid span:", bad_spans[0])
    raise ValueError("Fix invalid annotations before training.")

if boundary_mismatches:
    print("First text/offset mismatch:", boundary_mismatches[0])
    print("Some annotations may have non-identical entity_text; inspect before proceeding.")

BIO_LABELS = ["O"]
for label in ENTITY_LABELS:
    BIO_LABELS.extend([f"B-{label}", f"I-{label}"])

label2id = {label: i for i, label in enumerate(BIO_LABELS)}
id2label = {i: label for label, i in label2id.items()}

print("\nNumber of BIO classes:", len(BIO_LABELS))
print(BIO_LABELS)

In [ ]:
# CELL 6 — Restore the Phase 1 Transformer checkpoint

import torch
from pathlib import Path
from transformers import GPT2Config, GPT2LMHeadModel

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert DEVICE.type == "cuda", "Enable a Kaggle GPU before training."

checkpoint_candidates = list(
    Path("/kaggle/input").rglob("checkpoint_latest.pt")
)

# Prefer the known Transformer checkpoint dataset path if present.
preferred = [
    p for p in checkpoint_candidates
    if "cs781-phase-1-checkpoints" in str(p).lower()
]
if preferred:
    checkpoint_candidates = preferred

if not checkpoint_candidates:
    raise FileNotFoundError(
        "No checkpoint_latest.pt found under /kaggle/input. "
        "Attach your trained Phase 1 Transformer checkpoint dataset."
    )

loaded_checkpoint = None
CHECKPOINT_PATH = None

for path in checkpoint_candidates:
    try:
        candidate = torch.load(path, map_location="cpu", weights_only=False)
        state = candidate.get("model_state_dict", {})
        if any(k.startswith("transformer.wte.") for k in state):
            loaded_checkpoint = candidate
            CHECKPOINT_PATH = path
            break
    except Exception as exc:
        print("Could not inspect:", path, repr(exc))

if loaded_checkpoint is None:
    raise RuntimeError(
        "No GPT-2-style Transformer checkpoint was found. "
        "The attached checkpoint may belong to the RWKV architecture."
    )

print("Checkpoint:", CHECKPOINT_PATH)
print("Saved step:", loaded_checkpoint.get("step"))
print("Saved config:", loaded_checkpoint.get("model_config"))

saved_cfg = loaded_checkpoint.get("model_config", {})
vocab_size = int(saved_cfg.get("vocab_size", tokenizer.get_vocab_size()))
context_size = int(saved_cfg.get("context_length", 4096))
hidden_size = int(saved_cfg.get("hidden_size", 768))
num_layers = int(saved_cfg.get("num_layers", 12))
num_heads = int(saved_cfg.get("num_heads", 12))

assert vocab_size == tokenizer.get_vocab_size(), (
    f"Tokenizer/model vocabulary mismatch: {tokenizer.get_vocab_size()} vs {vocab_size}"
)

hf_config = GPT2Config(
    vocab_size=vocab_size,
    n_positions=context_size,
    n_ctx=context_size,
    n_embd=hidden_size,
    n_layer=num_layers,
    n_head=num_heads,
    n_inner=hidden_size * 4,
    resid_pdrop=0.0,
    embd_pdrop=0.0,
    attn_pdrop=0.0,
    use_cache=False,
)

phase1_lm = GPT2LMHeadModel(hf_config)
phase1_lm.load_state_dict(loaded_checkpoint["model_state_dict"], strict=True)
phase1_lm.config.use_cache = False

print("Restored Phase 1 model.")
print("Parameters (M):", round(sum(p.numel() for p in phase1_lm.parameters()) / 1e6, 2))

In [ ]:
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Random seed initialized.")

In [ ]:
# CELL 7 — Tokenize documents and align gold spans

from collections import defaultdict

MAX_LEN = 512
WINDOW_STRIDE = 448  # 64-token overlap
IGNORE_INDEX = -100

def document_to_windows(text, entities, max_len=MAX_LEN, stride=WINDOW_STRIDE):
    enc = tokenizer.encode(text, add_special_tokens=False)
    ids = enc.ids
    offsets = enc.offsets

    # Per-token labels, initially outside any entity.
    tags = [label2id["O"]] * len(ids)

    # Assign B-/I- tags based on token character offsets.
    for start, end, label, *rest in entities:
        start, end, label = int(start), int(end), str(label)
        overlapping = [
            i for i, (a, b) in enumerate(offsets)
            if b > a and a < end and b > start
        ]

        if not overlapping:
            continue

        # Require tokenization to cover the entity boundaries exactly.
        first, last = overlapping[0], overlapping[-1]
        if offsets[first][0] != start or offsets[last][1] != end:
            # Keep the example but report this condition for later analysis.
            pass

        tags[first] = label2id[f"B-{label}"]
        for i in overlapping[1:]:
            tags[i] = label2id[f"I-{label}"]

    windows = []
    for start_idx in range(0, len(ids), stride):
        end_idx = min(start_idx + max_len, len(ids))
        if start_idx >= end_idx:
            break

        # Token IDs plus aligned BIO labels.
        win_ids = ids[start_idx:end_idx]
        win_tags = tags[start_idx:end_idx]
        win_offsets = offsets[start_idx:end_idx]

        # Avoid training on an entity fragment cut by a window boundary.
        # For the first token in a window, an I-tag without preceding context
        # is converted to B-tag for that entity.
        if win_tags and id2label[win_tags[0]].startswith("I-"):
            ent_type = id2label[win_tags[0]][2:]
            win_tags[0] = label2id[f"B-{ent_type}"]

        windows.append({
            "input_ids": win_ids,
            "labels": win_tags,
            "offsets": win_offsets,
        })

        if end_idx == len(ids):
            break

    return windows

# Document-level holdout prevents windows from the same document leaking
# into both training and validation.
all_documents = {}
for fold_text, fold_gold in [
    (train_text_1, train_gold_1),
    (train_text_2, train_gold_2),
]:
    for doc_id, text in fold_text.items():
        if doc_id in all_documents:
            raise ValueError(f"Duplicate document ID across folds: {doc_id}")
        all_documents[doc_id] = (text, fold_gold[doc_id])

doc_ids = list(all_documents)
random.Random(SEED).shuffle(doc_ids)
n_val = max(1, int(0.1 * len(doc_ids)))
val_ids = set(doc_ids[:n_val])
train_ids = set(doc_ids[n_val:])

def make_windows(ids):
    result = []
    for doc_id in ids:
        text, entities = all_documents[doc_id]
        result.extend(document_to_windows(text, entities))
    return result

train_windows = make_windows(train_ids)
val_windows = make_windows(val_ids)

print("Train documents:", len(train_ids))
print("Validation documents:", len(val_ids))
print("Train windows:", len(train_windows))
print("Validation windows:", len(val_windows))
print("Example window length:", len(train_windows[0]["input_ids"]))

assert train_windows and val_windows
assert max(len(x["input_ids"]) for x in train_windows) <= MAX_LEN

In [ ]:
# CELL 8 — Token classification head + data loaders

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

class WindowDataset(Dataset):
    def __init__(self, windows):
        self.windows = windows

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        return self.windows[idx]

def collate_windows(items):
    batch_len = max(len(x["input_ids"]) for x in items)
    batch_size = len(items)

    input_ids = torch.full(
        (batch_size, batch_len), PAD_ID, dtype=torch.long
    )
    attention_mask = torch.zeros(
        (batch_size, batch_len), dtype=torch.long
    )
    labels = torch.full(
        (batch_size, batch_len), IGNORE_INDEX, dtype=torch.long
    )

    for i, item in enumerate(items):
        n = len(item["input_ids"])
        input_ids[i, :n] = torch.tensor(item["input_ids"])
        attention_mask[i, :n] = 1
        labels[i, :n] = torch.tensor(item["labels"])

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

BATCH_SIZE = 1
NUM_WORKERS = 0

train_loader_ner = DataLoader(
    WindowDataset(train_windows),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    collate_fn=collate_windows,
)

val_loader_ner = DataLoader(
    WindowDataset(val_windows),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=collate_windows,
)

class LegalNERModel(nn.Module):
    def __init__(self, pretrained_lm, num_labels):
        super().__init__()
        self.encoder = pretrained_lm.transformer
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(
            pretrained_lm.config.n_embd, num_labels
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
            return_dict=True,
        )
        hidden = self.dropout(outputs.last_hidden_state)
        return self.classifier(hidden)

ner_model = LegalNERModel(phase1_lm, len(BIO_LABELS)).to(DEVICE)

print("Device:", DEVICE)
print("Trainable parameters (M):", round(
    sum(p.numel() for p in ner_model.parameters() if p.requires_grad) / 1e6, 2
))
print("Train windows:", len(train_loader_ner))
print("Validation windows:", len(val_loader_ner))

# Smoke test before training.
sample_batch = next(iter(train_loader_ner))
with torch.no_grad():
    sample_logits = ner_model(
        sample_batch["input_ids"].to(DEVICE),
        sample_batch["attention_mask"].to(DEVICE),
    )
print("Smoke-test logits shape:", tuple(sample_logits.shape))
assert sample_logits.shape[-1] == len(BIO_LABELS)

In [ ]:
# CELL 9 — Fine-tune Phase 1 Transformer for legal NER

import math, time, json
from pathlib import Path
from sklearn.metrics import f1_score

NER_DIR = Path("/kaggle/working/phase2_ner")
NER_DIR.mkdir(parents=True, exist_ok=True)
BEST_NER_PATH = NER_DIR / "best_ner_model.pt"

EPOCHS = 2
LR = 2e-5
WEIGHT_DECAY = 0.01
GRAD_ACCUM = 8
MAX_GRAD_NORM = 1.0
USE_AMP = True

optimizer_ner = torch.optim.AdamW(
    ner_model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)
scaler_ner = torch.amp.GradScaler(
    "cuda", enabled=(DEVICE.type == "cuda" and USE_AMP)
)
criterion = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)

def evaluate_token_f1():
    ner_model.eval()
    gold_all, pred_all = [], []
    total_loss, steps = 0.0, 0

    with torch.no_grad():
        for batch in val_loader_ner:
            ids = batch["input_ids"].to(DEVICE)
            mask = batch["attention_mask"].to(DEVICE)
            gold = batch["labels"].to(DEVICE)

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=(DEVICE.type == "cuda" and USE_AMP),
            ):
                logits = ner_model(ids, mask)
                loss = criterion(
                    logits.reshape(-1, len(BIO_LABELS)),
                    gold.reshape(-1),
                )

            pred = logits.argmax(dim=-1)
            valid = gold != IGNORE_INDEX

            gold_all.extend(gold[valid].cpu().tolist())
            pred_all.extend(pred[valid].cpu().tolist())
            total_loss += loss.item()
            steps += 1

    # Report macro F1 over entity BIO classes; exclude O from macro averaging.
    entity_ids = [
        idx for idx, label in id2label.items()
        if label != "O"
    ]
    score = f1_score(
        gold_all,
        pred_all,
        labels=entity_ids,
        average="macro",
        zero_division=0,
    ) if gold_all else 0.0

    return total_loss / max(steps, 1), score

for epoch in range(1, EPOCHS + 1):
    ner_model.train()
    optimizer_ner.zero_grad(set_to_none=True)
    running_loss = 0.0
    start_time = time.time()

    for step, batch in enumerate(train_loader_ner, start=1):
        ids = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        gold = batch["labels"].to(DEVICE)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(DEVICE.type == "cuda" and USE_AMP),
        ):
            logits = ner_model(ids, mask)
            loss = criterion(
                logits.reshape(-1, len(BIO_LABELS)),
                gold.reshape(-1),
            )
            scaled_loss = loss / GRAD_ACCUM

        scaler_ner.scale(scaled_loss).backward()
        running_loss += loss.item()

        if step % GRAD_ACCUM == 0 or step == len(train_loader_ner):
            scaler_ner.unscale_(optimizer_ner)
            torch.nn.utils.clip_grad_norm_(
                ner_model.parameters(), MAX_GRAD_NORM
            )
            scaler_ner.step(optimizer_ner)
            scaler_ner.update()
            optimizer_ner.zero_grad(set_to_none=True)

        if step % 100 == 0:
            print(
                f"epoch={epoch}/{EPOCHS} step={step}/{len(train_loader_ner)} "
                f"loss={running_loss / 100:.4f}",
                flush=True,
            )
            running_loss = 0.0

    val_loss, val_macro_f1 = evaluate_token_f1()
    elapsed = (time.time() - start_time) / 60

    print(
        f"Epoch {epoch}: val_loss={val_loss:.4f} "
        f"val_token_macro_f1={val_macro_f1:.4f} "
        f"time={elapsed:.1f} min",
        flush=True,
    )

    if not BEST_NER_PATH.exists() or val_macro_f1 > best_val_f1:
        best_val_f1 = val_macro_f1
        torch.save({
            "model_state_dict": ner_model.state_dict(),
            "bio_labels": BIO_LABELS,
            "entity_labels": ENTITY_LABELS,
            "epoch": epoch,
            "val_token_macro_f1": val_macro_f1,
            "phase1_checkpoint": str(CHECKPOINT_PATH),
        }, BEST_NER_PATH)
        print("Saved best NER checkpoint:", BEST_NER_PATH)

print("Best validation token macro-F1:", best_val_f1)

In [ ]:
# CELL 10 — Fold 3 inference and submission JSON

import json
import torch
from pathlib import Path

# Load best model weights.
saved_ner = torch.load(
    BEST_NER_PATH,
    map_location=DEVICE,
    weights_only=False,
)
ner_model.load_state_dict(saved_ner["model_state_dict"])
ner_model.eval()

def decode_bio_spans(predicted_ids, offsets, text):
    spans = []
    current_label = None
    current_start = None
    current_end = None

    def close_span():
        if current_label is not None and current_start is not None:
            if current_end > current_start:
                spans.append([current_start, current_end, current_label])

    for pred_id, (start, end) in zip(predicted_ids, offsets):
        if end <= start:
            continue

        tag = id2label[int(pred_id)]

        if tag == "O":
            close_span()
            current_label = current_start = current_end = None
            continue

        prefix, entity_label = tag.split("-", 1)

        if prefix == "B" or current_label != entity_label:
            close_span()
            current_label = entity_label
            current_start = start
            current_end = end
        else:
            # Continue the same entity.
            current_end = end

    close_span()

    # Preserve valid character offsets and exact text slices.
    valid = []
    for start, end, label in spans:
        if 0 <= start < end <= len(text):
            valid.append([int(start), int(end), label])
    return valid

def predict_document(text):
    enc = tokenizer.encode(text, add_special_tokens=False)
    all_ids = enc.ids
    all_offsets = enc.offsets
    predictions = []

    # Non-overlapping windows prevent duplicate predictions from overlap.
    for start_idx in range(0, len(all_ids), MAX_LEN):
        end_idx = min(start_idx + MAX_LEN, len(all_ids))
        ids = all_ids[start_idx:end_idx]
        offsets = all_offsets[start_idx:end_idx]

        if not ids:
            continue

        input_tensor = torch.tensor([ids], dtype=torch.long, device=DEVICE)
        attention_mask = torch.ones_like(input_tensor)

        with torch.no_grad():
            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=(DEVICE.type == "cuda"),
            ):
                logits = ner_model(input_tensor, attention_mask)

        pred_ids = logits.argmax(dim=-1)[0].cpu().tolist()
        predictions.extend(
            decode_bio_spans(pred_ids, offsets, text)
        )

    # Sort by position, then remove exact duplicates if any.
    predictions.sort(key=lambda x: (x[0], x[1], x[2]))
    deduplicated = []
    seen = set()

    for span in predictions:
        key = tuple(span)
        if key not in seen:
            seen.add(key)
            deduplicated.append(span)

    return deduplicated

prediction_path = Path("/kaggle/working/prediction.json")
phase2_predictions = {}

for i, (doc_id, text) in enumerate(test_text.items(), start=1):
    phase2_predictions[str(doc_id)] = predict_document(text)

    if i % 10 == 0 or i == len(test_text):
        print(f"Predicted {i}/{len(test_text)} documents", flush=True)

with prediction_path.open("w", encoding="utf-8") as f:
    json.dump(phase2_predictions, f, ensure_ascii=False)

print("\nSaved:", prediction_path)
print("Documents:", len(phase2_predictions))
print("Entities predicted:", sum(len(v) for v in phase2_predictions.values()))

# Output sanity checks.
assert list(phase2_predictions.keys()) == [str(k) for k in test_text.keys()]

for doc_id, spans in phase2_predictions.items():
    text = test_text[doc_id]
    for span in spans:
        assert len(span) == 3
        start, end, label = span
        assert 0 <= start < end <= len(text)
        assert label in ENTITY_LABELS

print("Output validation passed.")
print("Example:", next(iter(phase2_predictions.items())))

In [ ]:

import torch
from pathlib import Path

CHECKPOINT_PATH = Path("/kaggle/working/phase2_ner/best_ner_model.pt")
PREDICTION_PATH = Path("/kaggle/working/prediction.json")

assert CHECKPOINT_PATH.exists(), f"Missing checkpoint: {CHECKPOINT_PATH}"
assert "model" in globals(), "The trained NER model is not in memory. Rerun the model-loading cell."

print("Model class:", type(model).__name__)
print("Device:", next(model.parameters()).device)
print("Parameters (M):", round(sum(p.numel() for p in model.parameters()) / 1e6, 2))
print("Tokenizer available:", "tokenizer" in globals())
print("Best checkpoint:", CHECKPOINT_PATH.exists())
print("Test prediction file:", PREDICTION_PATH.exists())

ckpt = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
print("Saved epoch:", ckpt.get("epoch"))
print("Saved validation token Macro-F1:", ckpt.get("val_token_macro_f1"))
print("Saved Phase 1 checkpoint:", ckpt.get("phase1_checkpoint"))
print("Saved BIO classes:", len(ckpt["bio_labels"]))


In [ ]:

import json
from pathlib import Path

DATA_DIR = Path("/kaggle/working/phase2_lner/Phase_2_LNER/Train")

with open(DATA_DIR / "fold2-text.json", encoding="utf-8") as f:
    val_texts = json.load(f)

with open(DATA_DIR / "fold2-gold.json", encoding="utf-8") as f:
    val_gold = json.load(f)

print("Validation documents:", len(val_texts))
print("Gold documents:", len(val_gold))
print("Matching document IDs:", set(val_texts) == set(val_gold))

assert set(val_texts) == set(val_gold), "Text and gold document IDs differ."

print("\nFirst document ID:", next(iter(val_texts)))
print("Text length:", len(val_texts[next(iter(val_texts))]))
print("Gold sample:", val_gold[next(iter(val_gold))][:3])


In [ ]:

import numpy as np
import torch

BIO_LABELS = ckpt["bio_labels"]
ENTITY_LABELS = ckpt["entity_labels"]
LABEL_TO_ID = {label: i for i, label in enumerate(BIO_LABELS)}

MAX_LEN = 512
STRIDE = 448
DEVICE = next(model.parameters()).device

def extract_logits(output):
    # Supports common custom-model return formats.
    if hasattr(output, "logits"):
        return output.logits
    if isinstance(output, dict):
        if "logits" in output:
            return output["logits"]
    if isinstance(output, (tuple, list)):
        for item in output:
            if torch.is_tensor(item) and item.ndim == 3:
                return item
    if torch.is_tensor(output) and output.ndim == 3:
        return output
    raise TypeError(
        f"Cannot extract token logits from output type {type(output)}"
    )

def decode_bio_to_spans(token_offsets, tag_ids, bio_labels):
    spans = []
    active_label = None
    active_start = None
    active_end = None

    def close_span():
        nonlocal active_label, active_start, active_end
        if active_label is not None and active_end > active_start:
            spans.append((active_start, active_end, active_label))
        active_label = active_start = active_end = None

    for (start, end), tag_id in zip(token_offsets, tag_ids):
        if start == end:  # Special/empty-offset token
            continue

        tag = bio_labels[int(tag_id)]
        if tag == "O":
            close_span()
            continue

        prefix, label = tag.split("-", 1)

        if prefix == "B" or active_label != label:
            close_span()
            active_label = label
            active_start = start
            active_end = end
        else:
            # Continue an existing I-label span.
            active_end = end

    close_span()
    return spans

@torch.inference_mode()
def predict_document_spans(text):
    encoded = tokenizer.encode(text, add_special_tokens=False)
    token_ids = encoded.ids
    offsets = encoded.offsets

    if not token_ids:
        return []

    # Aggregate logits from overlapping windows.
    logits_sum = np.zeros(
        (len(token_ids), len(BIO_LABELS)), dtype=np.float64
    )
    counts = np.zeros(len(token_ids), dtype=np.int32)

    was_training = model.training
    model.eval()

    try:
        for start in range(0, len(token_ids), STRIDE):
            end = min(start + MAX_LEN, len(token_ids))
            ids = torch.tensor(
                [token_ids[start:end]], dtype=torch.long, device=DEVICE
            )
            mask = torch.ones_like(ids)

            output = model(input_ids=ids, attention_mask=mask)
            logits = extract_logits(output)[0].float().cpu().numpy()

            expected_classes = len(BIO_LABELS)
            if logits.shape[-1] != expected_classes:
                raise ValueError(
                    f"Model emits {logits.shape[-1]} classes, "
                    f"but checkpoint has {expected_classes} BIO labels."
                )

            logits_sum[start:end] += logits[:end-start]
            counts[start:end] += 1

            if end == len(token_ids):
                break
    finally:
        model.train(was_training)

    mean_logits = logits_sum / np.maximum(counts[:, None], 1)
    predicted_ids = mean_logits.argmax(axis=-1)

    spans = decode_bio_to_spans(offsets, predicted_ids, BIO_LABELS)

    # Keep only spans inside the original document.
    return sorted({
        (int(s), int(e), label)
        for s, e, label in spans
        if 0 <= s < e <= len(text)
    })


In [ ]:

import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Build the NER wrapper around the existing Phase 1 GPT-2 model.
# `phase1_lm` should be the original pretrained GPT2LMHeadModel.
assert "phase1_lm" in globals(), (
    "phase1_lm is missing. Rerun the cell that restores the Phase 1 GPT-2 checkpoint."
)
assert "LegalNERModel" in globals(), (
    "LegalNERModel is missing. Rerun the cell that defines it."
)

model = LegalNERModel(
    pretrained_lm=phase1_lm,
    num_labels=len(ckpt["bio_labels"])
)

missing, unexpected = model.load_state_dict(
    ckpt["model_state_dict"],
    strict=True
)

model = model.to(DEVICE)
model.eval()

print("Loaded NER model:", type(model).__name__)
print("Device:", next(model.parameters()).device)
print("NER classes:", len(ckpt["bio_labels"]))
print("Strict checkpoint load successful.")


In [ ]:

# Use a short test sequence; no training or weight updates.
sample_ids = tokenizer.encode(
    next(iter(val_texts.values()))[:1000],
    add_special_tokens=False
).ids[:64]

input_ids = torch.tensor([sample_ids], dtype=torch.long, device=DEVICE)
attention_mask = torch.ones_like(input_ids)

with torch.inference_mode():
    logits = model(
        input_ids=input_ids,
        attention_mask=attention_mask
    )

print("Input shape:", tuple(input_ids.shape))
print("NER logits shape:", tuple(logits.shape))

assert logits.shape[-1] == len(ckpt["bio_labels"]), (
    f"Expected {len(ckpt['bio_labels'])} NER classes, got {logits.shape[-1]}"
)

print("PASS: model returns 25 NER classes.")


In [ ]:

# Inspect the tokenizer's exact character offsets on real gold entities.
from pathlib import Path

doc_id = next(iter(val_texts))
text = val_texts[doc_id]
gold_entities = val_gold[doc_id]

encoded = tokenizer.encode(text, add_special_tokens=False)

print("Document:", doc_id)
print("Text length:", len(text))
print("Token count:", len(encoded.ids))
print("\nFirst 12 gold entities and overlapping tokens:\n")

for start, end, label, entity_text in gold_entities[:12]:
    print(f"\nGOLD {label} [{start}:{end}] {text[start:end]!r}")

    overlaps = []
    for token_id, (ts, te) in zip(encoded.ids, encoded.offsets):
        if ts < end and te > start:
            overlaps.append((ts, te, text[ts:te], token_id))

    for ts, te, token_text, token_id in overlaps:
        print(f"  token [{ts}:{te}] {token_text!r} (id={token_id})")

print("\nTokenizer:", type(tokenizer).__name__)
print("First offsets:", encoded.offsets[:10])


In [ ]:

def decode_bio_to_spans(token_offsets, tag_ids, bio_labels, text):
    spans = []
    active_label = None
    active_start = None
    active_end = None

    def close_span():
        nonlocal active_label, active_start, active_end

        if active_label is not None:
            # Trim spaces introduced by byte-level BPE offsets.
            start = active_start
            end = active_end

            while start < end and text[start].isspace():
                start += 1
            while end > start and text[end - 1].isspace():
                end -= 1

            if start < end:
                spans.append((start, end, active_label))

        active_label = active_start = active_end = None

    for (start, end), tag_id in zip(token_offsets, tag_ids):
        if start == end:
            continue

        tag = bio_labels[int(tag_id)]

        if tag == "O":
            close_span()
            continue

        prefix, label = tag.split("-", 1)

        if prefix == "B" or active_label != label:
            close_span()
            active_label = label
            active_start = start
            active_end = end
        else:
            active_end = end

    close_span()
    return spans


In [ ]:

import numpy as np
import torch

BIO_LABELS = ckpt["bio_labels"]
ENTITY_LABELS = ckpt["entity_labels"]
DEVICE = next(model.parameters()).device

MAX_LEN = 512
STRIDE = 448

def decode_bio_to_spans(token_offsets, tag_ids, bio_labels, text):
    spans = []
    active_label = None
    active_start = None
    active_end = None

    def close_span():
        nonlocal active_label, active_start, active_end

        if active_label is not None:
            start, end = active_start, active_end

            while start < end and text[start].isspace():
                start += 1
            while end > start and text[end - 1].isspace():
                end -= 1

            if start < end:
                spans.append((int(start), int(end), active_label))

        active_label = active_start = active_end = None

    for (start, end), tag_id in zip(token_offsets, tag_ids):
        if start == end:
            continue

        tag = bio_labels[int(tag_id)]

        if tag == "O":
            close_span()
            continue

        prefix, label = tag.split("-", 1)

        if prefix == "B" or active_label != label:
            close_span()
            active_label = label
            active_start = start
            active_end = end
        else:
            active_end = end

    close_span()
    return spans


def extract_logits(output):
    if hasattr(output, "logits"):
        return output.logits
    if isinstance(output, dict) and "logits" in output:
        return output["logits"]
    if isinstance(output, (tuple, list)):
        for item in output:
            if torch.is_tensor(item) and item.ndim == 3:
                return item
    if torch.is_tensor(output) and output.ndim == 3:
        return output
    raise TypeError(f"Cannot extract logits from {type(output)}")


@torch.inference_mode()
def predict_document_spans(text):
    encoded = tokenizer.encode(text, add_special_tokens=False)
    token_ids = encoded.ids
    offsets = encoded.offsets

    if not token_ids:
        return []

    logits_sum = np.zeros(
        (len(token_ids), len(BIO_LABELS)), dtype=np.float64
    )
    counts = np.zeros(len(token_ids), dtype=np.int32)

    was_training = model.training
    model.eval()

    try:
        for start in range(0, len(token_ids), STRIDE):
            end = min(start + MAX_LEN, len(token_ids))

            ids = torch.tensor(
                [token_ids[start:end]],
                dtype=torch.long,
                device=DEVICE,
            )
            mask = torch.ones_like(ids)

            output = model(input_ids=ids, attention_mask=mask)
            logits = extract_logits(output)[0].float().cpu().numpy()

            if logits.shape[-1] != len(BIO_LABELS):
                raise ValueError(
                    f"Expected {len(BIO_LABELS)} classes, "
                    f"received {logits.shape[-1]}"
                )

            logits_sum[start:end] += logits[:end-start]
            counts[start:end] += 1

            if end == len(token_ids):
                break
    finally:
        model.train(was_training)

    mean_logits = logits_sum / np.maximum(counts[:, None], 1)
    predicted_ids = mean_logits.argmax(axis=-1)

    spans = decode_bio_to_spans(
        offsets, predicted_ids, BIO_LABELS, text
    )

    return sorted(set(
        (s, e, label)
        for s, e, label in spans
        if 0 <= s < e <= len(text)
    ))

print("Corrected inference function ready.")


In [ ]:

val_predictions = {}

for i, (doc_id, text) in enumerate(val_texts.items(), start=1):
    val_predictions[doc_id] = predict_document_spans(text)

    if i % 5 == 0 or i == len(val_texts):
        print(f"Processed {i}/{len(val_texts)} documents")

print("\nDocuments:", len(val_predictions))
print("Example predictions:", next(iter(val_predictions.values()))[:10])


In [ ]:

from collections import defaultdict

gold_by_label = defaultdict(set)
pred_by_label = defaultdict(set)

for doc_id, entities in val_gold.items():
    for start, end, label, *_ in entities:
        gold_by_label[label].add(
            (str(doc_id), int(start), int(end), label)
        )

for doc_id, entities in val_predictions.items():
    for start, end, label in entities:
        pred_by_label[label].add(
            (str(doc_id), int(start), int(end), label)
        )

scores = {}

print(
    f"{'Label':<12}{'Gold':>7}{'Pred':>7}{'TP':>7}"
    f"{'Prec':>9}{'Recall':>9}{'F1':>9}"
)

for label in ENTITY_LABELS:
    gold_set = gold_by_label[label]
    pred_set = pred_by_label[label]

    tp = len(gold_set & pred_set)
    fp = len(pred_set - gold_set)
    fn = len(gold_set - pred_set)

    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if precision + recall else 0.0
    )

    scores[label] = {
        "gold": len(gold_set),
        "predicted": len(pred_set),
        "TP": tp,
        "precision": precision,
        "recall": recall,
        "F1": f1,
    }

    print(
        f"{label:<12}{len(gold_set):>7}{len(pred_set):>7}{tp:>7}"
        f"{precision:>9.3f}{recall:>9.3f}{f1:>9.3f}"
    )

exact_macro_f1 = sum(
    scores[label]["F1"] for label in ENTITY_LABELS
) / len(ENTITY_LABELS)

print(f"\nExact-span entity Macro-F1: {exact_macro_f1:.4f}")


In [ ]:

for doc_id in list(val_texts.keys())[:3]:
    text = val_texts[doc_id]
    gold = [
        (int(s), int(e), label)
        for s, e, label, *_ in val_gold[doc_id]
    ]
    pred = val_predictions[doc_id]

    print(f"\n{'=' * 70}")
    print("Document:", doc_id)
    print(f"Gold: {len(gold)} | Predicted: {len(pred)}")

    print("\nGOLD:")
    for s, e, label in sorted(gold)[:12]:
        print(f"{label:<12} [{s}:{e}] {text[s:e]!r}")

    print("\nPREDICTED:")
    for s, e, label in sorted(pred)[:12]:
        print(f"{label:<12} [{s}:{e}] {text[s:e]!r}")


In [ ]:

import torch
from collections import Counter

ckpt = torch.load(
    "/kaggle/working/phase2_ner/best_ner_model.pt",
    map_location="cpu",
    weights_only=False
)

print("Saved epoch:", ckpt["epoch"])
print("Saved token Macro-F1:", ckpt["val_token_macro_f1"])

# Inspect classifier bias and weight magnitudes.
state = ckpt["model_state_dict"]
W = state["classifier.weight"].float()
b = state["classifier.bias"].float()
labels = ckpt["bio_labels"]

print("\nClassifier bias by class:")
for i, label in enumerate(labels):
    print(f"{label:<15} bias={b[i].item(): .4f} weight_norm={W[i].norm().item():.3f}")

# Count predictions on one validation document.
text = next(iter(val_texts.values()))
enc = tokenizer.encode(text, add_special_tokens=False)
ids = torch.tensor([enc.ids[:512]], dtype=torch.long, device=DEVICE)
mask = torch.ones_like(ids)

model.eval()
with torch.inference_mode():
    logits = model(input_ids=ids, attention_mask=mask)
    pred_ids = logits.argmax(-1)[0].cpu().tolist()

counts = Counter(labels[i] for i in pred_ids)
print("\nPredicted BIO token counts for first 512 tokens:")
for label, count in counts.most_common():
    print(f"{label:<15} {count}")


In [ ]:

from collections import Counter

stats = Counter()
examples = []

for doc_id, text in val_texts.items():
    enc = tokenizer.encode(text, add_special_tokens=False)

    for start, end, label, entity_text in val_gold[doc_id]:
        overlapping = [
            (ts, te) for ts, te in enc.offsets
            if ts < end and te > start
        ]

        if not overlapping:
            stats["no_overlap"] += 1
            continue

        first_start, first_end = overlapping[0]
        last_start, last_end = overlapping[-1]

        stats["entities"] += 1
        if first_start < start or last_end > end:
            stats["boundary_inside_token"] += 1
            if len(examples) < 12:
                examples.append({
                    "doc_id": doc_id,
                    "gold": (start, end, label, entity_text),
                    "first_token": (first_start, first_end, text[first_start:first_end]),
                    "last_token": (last_start, last_end, text[last_start:last_end]),
                })
        else:
            stats["token_boundaries_cover_entity"] += 1

print("Alignment summary:", dict(stats))
print("\nBoundary examples:")
for item in examples:
    print(item)


In [ ]:

from collections import Counter

stats = Counter()
unrepresentable = []

for doc_id, text in val_texts.items():
    enc = tokenizer.encode(text, add_special_tokens=False)

    for start, end, label, entity_text in val_gold[doc_id]:
        # Token offsets fully inside the gold entity.
        inside = [
            (ts, te) for ts, te in enc.offsets
            if start <= ts and te <= end and ts < te
        ]

        # Exact coverage means a contiguous token sequence starts and ends
        # exactly at the gold character boundaries.
        exact = bool(inside) and inside[0][0] == start and inside[-1][1] == end

        if exact:
            stats["exactly_representable"] += 1
        else:
            stats["not_exactly_representable"] += 1
            if len(unrepresentable) < 15:
                unrepresentable.append(
                    (doc_id, start, end, label, entity_text, inside[:3])
                )

total = stats["exactly_representable"] + stats["not_exactly_representable"]

print("Gold entities:", total)
print("Exactly representable:", stats["exactly_representable"])
print("Not exactly representable:", stats["not_exactly_representable"])
print("Representable fraction:",
      round(stats["exactly_representable"] / max(total, 1), 4))

print("\nExamples not exactly representable:")
for item in unrepresentable:
    print(item)


In [ ]:

from collections import defaultdict
import numpy as np
import torch

def gold_char_labels_to_token_labels(text, gold_entities, offsets):
    # Diagnostic only: assign each token the label with the largest
    # character overlap. This does not alter the training checkpoint.
    labels = ["O"] * len(offsets)

    for start, end, label, *_ in gold_entities:
        for i, (ts, te) in enumerate(offsets):
            overlap = max(0, min(te, end) - max(ts, start))
            if overlap > 0:
                labels[i] = label

    return labels

token_tp = defaultdict(int)
token_fp = defaultdict(int)
token_fn = defaultdict(int)

for doc_id, text in val_texts.items():
    enc = tokenizer.encode(text, add_special_tokens=False)
    ids = enc.ids
    offsets = enc.offsets

    gold_labels = gold_char_labels_to_token_labels(
        text, val_gold[doc_id], offsets
    )

    predicted_labels = []
    model.eval()

    with torch.inference_mode():
        for start in range(0, len(ids), 448):
            end = min(start + 512, len(ids))
            input_ids = torch.tensor(
                [ids[start:end]], device=DEVICE, dtype=torch.long
            )
            mask = torch.ones_like(input_ids)
            logits = extract_logits(
                model(input_ids=input_ids, attention_mask=mask)
            )[0]
            predicted_labels.extend(
                logits.argmax(-1).cpu().tolist()
            )
            if end == len(ids):
                break

    predicted_labels = [BIO_LABELS[i] for i in predicted_labels]

    for label in ENTITY_LABELS:
        gold_set = {
            i for i, value in enumerate(gold_labels) if value == label
        }
        pred_set = {
            i for i, value in enumerate(predicted_labels)
            if value.endswith("-" + label)
        }

        token_tp[label] += len(gold_set & pred_set)
        token_fp[label] += len(pred_set - gold_set)
        token_fn[label] += len(gold_set - pred_set)

print("Token-level type-only diagnostic (not the official metric)")
print(f"{'Label':<12}{'Precision':>12}{'Recall':>12}{'F1':>10}")

type_f1s = []
for label in ENTITY_LABELS:
    tp, fp, fn = token_tp[label], token_fp[label], token_fn[label]
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * p * r / (p + r) if p + r else 0.0
    type_f1s.append(f1)
    print(f"{label:<12}{p:>12.3f}{r:>12.3f}{f1:>10.3f}")

print("\nToken-level type-only Macro-F1:", round(float(np.mean(type_f1s)), 4))
